# Chapter 2: Being Clear and Direct

**Technique:** Clear and direct instruction

**Contents**
* [Lesson: Explicit Instructions](#lesson)
* [Exercises](#exercises)
* [Example Playground](#example-playground)

In [ ]:
import { getCompletion, MODEL, client } from "../lib/helpers.js";

## Lesson: Explicit Instructions

Claude responds best to explicit, direct instructions. When a prompt is vague, Claude fills in the gaps with reasonable defaults, but those defaults rarely match your production requirements.

**State exactly what you want.** Think of a well written Jira ticket or a precise code comment: you'd specify the format, the scope, the expected output, and the audience. The same discipline applies to prompts.

Three dimensions to make explicit:

* **Format**: vague is "tell me about X"; precise is "give me 3 bullet points about X".
* **Length**: vague is "write something about Y"; precise is "write a 400 word design doc about Y".
* **Audience**: vague is "explain Z"; precise is "explain Z to a developer unfamiliar with distributed systems".

A vague prompt forces Claude to guess. A precise prompt is a contract: Claude knows exactly what deliverable to produce. In a production pipeline where output is parsed, logged, or fed into another model, ambiguity is a bug, not a style choice.

**You're not being rude by being direct.** Dropping filler phrases like "could you possibly" and "if you don't mind" does not change how Claude responds to instructions. Claude is not a human colleague; it does not infer goodwill from hedging language. Every token spent on pleasantries is a token that could carry a precise constraint.

In [ ]:
// Vague prompt, Claude must guess format, length, and depth
const vagueResponse = await getCompletion("tell me about errors");
console.log("=== VAGUE PROMPT OUTPUT ===");
console.log(vagueResponse);

// Direct prompt with explicit format, count, and scope
const directResponse = await getCompletion(
  "List 3 common Node.js async error-handling mistakes as bullet points. " +
  "Each bullet: one sentence on what the mistake is, one sentence on how to fix it."
);
console.log("\n=== DIRECT PROMPT OUTPUT ===");
console.log(directResponse);

## Exercises

### Exercise 2.1: Spanish language support response

**Scenario**: your SaaS product serves Spanish speaking users. A customer submits a support ticket asking how to reset their password. You need Claude to answer in Spanish so you can route it directly to the customer without a translation step.

**Task**: Edit `PROMPT` so that Claude's response is in Spanish.

**Success criteria**: the response contains at least one common Spanish word (e.g. `hola`, `gracias`, `buenos`, `cómo`, `ayudar`).

In [ ]:
import { includesAny, grade } from "../lib/grading.js";

const PROMPT = "Eres un agente de soporte. Responde en español, empezando con un saludo. ¿Cómo restablezco mi contraseña?";

const response = await getCompletion(PROMPT);
const gradeExercise = (text) => includesAny(text, ["hola", "gracias", "buenos", "cómo", "ayudar"]);
grade(response, gradeExercise(response));

In [ ]:
import { exercise_2_1_hint } from "../hints.js";
console.log(exercise_2_1_hint);

### Exercise 2.2: Parseable single word answer

**Scenario**: you're writing a shell script that selects a test framework for a new Node.js project. You want to call Claude and `grep` the output directly, with no JSON parsing and no stripping prose. The answer must be exactly one word so your script can consume it.

**Task**: Edit `PROMPT` so that Claude replies with ONLY the name of the most downloaded JavaScript unit testing framework, with nothing else: no punctuation, no explanation.

**Success criteria**: `response.trim()` equals exactly `"Jest"`.

In [ ]:
import { equalsExact, grade } from "../lib/grading.js";

const PROMPT = "What is the most downloaded JavaScript unit-testing framework? Reply with only its name as a single word, with no punctuation, no quotes, and no explanation.";

const response = await getCompletion(PROMPT);
const gradeExercise = (text) => equalsExact(text.trim(), "Jest");
grade(response, gradeExercise(response));

In [ ]:
import { exercise_2_2_hint } from "../hints.js";
console.log(exercise_2_2_hint);

### Exercise 2.3: Full length design document

**Scenario**: your team's RFC process requires design documents of at least 800 words before a proposal can be scheduled for review. A stub or a summary is rejected by the tooling. You need Claude to produce a real draft, not a placeholder.

**Task**: Edit `PROMPT` so that Claude writes a design document of at least 800 words. The topic is up to you, so pick any engineering system (e.g. a rate limiter, a distributed cache, a job queue).

**Success criteria**: the word count of the response is ≥ 800.

In [ ]:
import { wordCount, grade } from "../lib/grading.js";

const PROMPT = "Write a detailed software design document of at least 1000 words for a distributed rate limiter. Include these sections, each several paragraphs long: Overview, Goals and Non-Goals, Architecture, Data Model, Algorithms (token bucket and sliding window), Public API, Failure Modes, Scaling, Observability, and Rollout Plan.";

const response = await getCompletion(PROMPT);
const gradeExercise = (text) => wordCount(text) >= 800;
grade(response, gradeExercise(response));

In [ ]:
import { exercise_2_3_hint } from "../hints.js";
console.log(exercise_2_3_hint);

## Common mistakes

**Asking for quality adjectives instead of measurable constraints**
"Be detailed" and "be thorough" are not instructions, they're hopes. Claude will produce something it considers detailed, which may be 80 words. Give a hard floor: "write at least 800 words" is a constraint; "be thorough" is not.

**Not specifying output language or locale**
If your users are French speaking and your prompt is in English, Claude defaults to English. Specify the output language explicitly: "respond in French" is one phrase that eliminates an entire class of routing bugs.

**Not specifying output format for machine consumption**
If a downstream script parses Claude's output, treat it like an API contract. "Reply with only the status code, no other text" is a format constraint. Without it, Claude may write a sentence explaining the code, and your `parseInt` call breaks.

**Stretch challenge**: Write a prompt that reliably produces exactly a 5 item numbered list, no more and no fewer items. Verify by counting lines in the output.

**Reflect**: Each retry in a production pipeline costs latency, tokens, and money. How does adding explicit format and length constraints to a prompt reduce the retry rate? What would a prompt quality checklist look like for your team's use case?

## Example Playground

Use the cell below to experiment freely with direct vs. vague prompts. The graders are not active here.

In [ ]:
// Try your own prompt: experiment with explicit format, length, and audience constraints
const PROMPT = "In exactly 3 bullet points, list the trade-offs between REST and gRPC for internal microservice communication.";
console.log(await getCompletion(PROMPT));